# EventHub AI — Pipeline 06: Continued QLoRA Fine-Tuning (V2 -> V3) & Benchmark

Mục tiêu: **Huấn luyện bổ sung (Continued Fine-Tuning)** từ adapter đã train trước (`adapters/eventhub_qwen_v2`), **KHÔNG** reset về base model.
- Bắt buộc nạp trọng số PEFT V2 sẵn có bằng `PeftModel.from_pretrained(..., is_trainable=True)` (nếu không tìm thấy adapter V2 sẽ báo lỗi ngay).
- Kết hợp 91 tài liệu mới (`Test_doc`), 10 tài liệu regression cũ (`OLD`), 8 mẫu diverse V2, và Replay Archive V2 (lọc trùng title với val/test -> 0 data leakage) để chống catastrophic forgetting.
- Học: `lr=5e-5`, `epochs=3`, `cosine`, `warmup_ratio=0.08`, `eval_strategy="epoch"`, `load_best_model_at_end=True`.
- Đánh giá Pre-V2 vs Post-V3 trên cùng tập holdout `test_sft.jsonl` (22 tài liệu chưa từng thấy khi train, gồm cả 2 tài liệu regression cũ) rồi merge LoRA -> GGUF `Q4_K_M` cho Ollama.


In [ ]:
# [1] Cài đặt dependencies cho Continued QLoRA SFT & Evaluation
# Gỡ torchaudio/torchvision/torchao (không dùng cho text LLM) để tránh lỗi lệch phiên bản CUDA/torchao.
!pip uninstall -y -q torchaudio torchvision torchao
!pip install -q -U transformers datasets peft bitsandbytes trl accelerate sentencepiece "protobuf>=5.29.1,<7.0.0"

import os, re, json, random, inspect, collections
import torch, transformers, peft, bitsandbytes as bnb
print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | PEFT: {peft.__version__} | CUDA: {torch.cuda.is_available()}")


In [ ]:
# [2] Kiểm tra GPU & thiết lập tham số an toàn cho Continued Fine-Tuning
if not torch.cuda.is_available():
    raise RuntimeError("Yêu cầu GPU CUDA (T4 / L4 / A100) để chạy Continued QLoRA.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {device_name} ({vram_gb:.2f} GB VRAM)")

if vram_gb >= 22:  # L4 / A100 (24GB+)
    BATCH_SIZE, GRAD_ACCUM, MAX_SEQ_LENGTH = 2, 4, 6144
    BF16_MODE = torch.cuda.is_bf16_supported()
    FP16_MODE = not BF16_MODE
elif vram_gb >= 14:  # T4 / V100 (15-16GB)
    BATCH_SIZE, GRAD_ACCUM, MAX_SEQ_LENGTH = 1, 8, 5120
    BF16_MODE, FP16_MODE = False, True
else:
    BATCH_SIZE, GRAD_ACCUM, MAX_SEQ_LENGTH = 1, 8, 4096
    BF16_MODE, FP16_MODE = False, True

LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
WARMUP_RATIO = 0.08
REPLAY_RATIO = 0.4
SEED = 42
print(f"Config -> Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM}, EffectiveBatch={BATCH_SIZE*GRAD_ACCUM}, MaxSeq={MAX_SEQ_LENGTH}, LR={LEARNING_RATE}, Epochs={NUM_EPOCHS}")


In [ ]:
# [3] Mount Google Drive & Kiểm tra nghiêm ngặt Adapter V2 đã huấn luyện trước đó (TUYỆT ĐỐI KHÔNG TRAIN TỪ BASE)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
REPO_DATASET_DIR = '/content/EventHub/ai/datasets/sft_continued_v3'
DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, 'datasets/sft_continued_v3')
DATA_DIR = DRIVE_DATASET_DIR if os.path.exists(os.path.join(DRIVE_DATASET_DIR, 'train_sft.jsonl')) else REPO_DATASET_DIR

if not os.path.exists(os.path.join(DATA_DIR, 'train_sft.jsonl')):
    raise FileNotFoundError(
        "Không tìm thấy sft_continued_v3 (train_sft.jsonl). Hãy copy thư mục ai/datasets/sft_continued_v3 lên Google Drive "
        "tại EventHub_AI/datasets/sft_continued_v3 hoặc clone repo vào /content/EventHub."
    )

CANDIDATE_V2_ADAPTERS = [
    os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_v2'),
    os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_dpo_final'),
    os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_sft'),
]
PARENT_ADAPTER_DIR = None
for cand in CANDIDATE_V2_ADAPTERS:
    has_cfg = os.path.isfile(os.path.join(cand, 'adapter_config.json'))
    has_w = os.path.isfile(os.path.join(cand, 'adapter_model.safetensors')) or os.path.isfile(os.path.join(cand, 'adapter_model.bin'))
    if has_cfg and has_w:
        PARENT_ADAPTER_DIR = cand
        break

if PARENT_ADAPTER_DIR is None:
    raise FileNotFoundError(
        "DỪNG AN TOÀN: Không tìm thấy Adapter V2 đã huấn luyện (adapter_config.json + adapter_model.safetensors) trong "
        + ", ".join(CANDIDATE_V2_ADAPTERS)
        + ". Continued Fine-Tuning nghiêm cấm reset về Base Model!"
    )

with open(os.path.join(PARENT_ADAPTER_DIR, 'adapter_config.json'), 'r', encoding='utf-8') as f:
    PARENT_ADAPTER_CFG = json.load(f)
BASE_MODEL_ID = PARENT_ADAPTER_CFG.get('base_model_name_or_path') or 'Qwen/Qwen3-4B'

V2_REPLAY_FILE = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2/sft/train_sft.jsonl')
V3_CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints/eventhub_qwen_v3')
V3_ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_v3')
V3_MERGED_DIR = os.path.join(DRIVE_ROOT, 'merged/eventhub_qwen_v3_fp16')
V3_EXPORT_DIR = os.path.join(DRIVE_ROOT, 'export')
for d in [V3_CHECKPOINT_DIR, V3_ADAPTER_DIR, V3_EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Dữ liệu V3        : {DATA_DIR}")
print(f"✅ Parent Adapter V2 : {PARENT_ADAPTER_DIR}")
print(f"✅ Base Model Config : {BASE_MODEL_ID} (r={PARENT_ADAPTER_CFG.get('r')}, alpha={PARENT_ADAPTER_CFG.get('lora_alpha')})")
print(f"✅ Optional Replay   : {V2_REPLAY_FILE} (tồn tại={os.path.isfile(V2_REPLAY_FILE)})")


In [ ]:
# [4] Nạp dataset sft_continued_v3 + Pha trộn Replay V2 (loại bỏ mọi title trùng với Validation/Test để 0 leakage)
from datasets import Dataset

def norm_title_key(t):
    return re.sub(r"[^a-z0-9à-ỹ]+", " ", (t or "").lower()).strip()

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(os.path.join(DATA_DIR, 'train_sft.jsonl'))
val_rows = load_jsonl(os.path.join(DATA_DIR, 'validation_sft.jsonl'))
test_rows = load_jsonl(os.path.join(DATA_DIR, 'test_sft.jsonl'))

train_titles = {norm_title_key(json.loads(r['messages'][-1]['content']).get('title')) for r in train_rows}
val_titles = {norm_title_key(json.loads(r['messages'][-1]['content']).get('title')) for r in val_rows}
test_titles = {norm_title_key(json.loads(r['messages'][-1]['content']).get('title')) for r in test_rows}
assert not (train_titles & val_titles) and not (train_titles & test_titles) and not (val_titles & test_titles), "Phát hiện trùng tiêu đề giữa các split!"

blocked_eval_titles = val_titles | test_titles
external_replay_added = 0
if os.path.isfile(V2_REPLAY_FILE) and REPLAY_RATIO > 0:
    v2_raw = load_jsonl(V2_REPLAY_FILE)
    eligible = []
    seen = set(train_titles)
    for row in v2_raw:
        try:
            tkey = norm_title_key(json.loads(row['messages'][-1]['content']).get('title'))
        except Exception:
            continue
        if tkey and (tkey in blocked_eval_titles or tkey in seen):
            continue
        eligible.append(row)
        if tkey:
            seen.add(tkey)
    rng = random.Random(SEED)
    target_n = min(len(eligible), int(round(len(train_rows) * REPLAY_RATIO)))
    chosen = rng.sample(eligible, target_n) if target_n < len(eligible) else eligible
    train_rows = train_rows + chosen
    rng.shuffle(train_rows)
    external_replay_added = len(chosen)

print(f"Train samples: {len(train_rows)} (trong đó đã có 15 replay nhúng + {external_replay_added} historical V2 replay)")
print(f"Val samples  : {len(val_rows)}")
print(f"Test holdout : {len(test_rows)} (không xuất hiện trong train/replay)")


In [ ]:
# [5] Nạp Base Model (4-bit NF4) & Tiếp tục từ Trọng số PEFT V2 (PeftModel.from_pretrained, is_trainable=True)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if BF16_MODE else torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if BF16_MODE else torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
base_model.config.use_cache = False

# CRITICAL: Không dùng get_peft_model(); bắt buộc resume trọng số V2 đã train!
model = PeftModel.from_pretrained(base_model, PARENT_ADAPTER_DIR, is_trainable=True)
if not BF16_MODE:
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.to(torch.float32)
        elif p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float16)
model.print_trainable_parameters()


In [ ]:
# [6] Định nghĩa bộ đo Evaluation chuẩn EventHub & Chạy Pre-Training Baseline (V2 Adapter trên 22 mẫu test holdout)
def render_prompt(messages, add_generation_prompt=False):
    sig = inspect.signature(tokenizer.apply_chat_template)
    kw = {'tokenize': False, 'add_generation_prompt': add_generation_prompt}
    if 'enable_thinking' in sig.parameters:
        kw['enable_thinking'] = False
    return tokenizer.apply_chat_template(messages, **kw)

def strip_fences(txt):
    s = txt.strip()
    s = re.sub(r'^<think>.*?</think>\s*', '', s, flags=re.S).strip()
    if s.startswith('```'):
        s = re.sub(r'^```(?:json)?\s*', '', s)
        s = re.sub(r'\s*```$', '', s)
    m = re.search(r'\{.*\}', s, flags=re.S)
    return m.group(0) if m else s

def norm_str(x):
    return re.sub(r'\s+', ' ', str(x or '').strip().lower())

def f1_multiset(pred_keys, gold_keys):
    if not pred_keys and not gold_keys:
        return 1.0
    if not pred_keys or not gold_keys:
        return 0.0
    pc, gc = collections.Counter(pred_keys), collections.Counter(gold_keys)
    inter = sum((pc & gc).values())
    return (2.0 * inter) / (len(pred_keys) + len(gold_keys))

def evaluate_model_on_holdout(eval_model, rows, tag, max_new_tokens=1536):
    eval_model.eval()
    eval_model.config.use_cache = True
    metrics = collections.Counter()
    session_f1s, ticket_f1s = [], []
    null_venue_total, null_venue_halluc = 0, 0
    zero_tkt_total, zero_tkt_halluc = 0, 0
    details = []

    for idx, row in enumerate(rows):
        gold = json.loads(row['messages'][-1]['content'])
        prompt = render_prompt(row['messages'][:-1], add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH).to(eval_model.device)
        with torch.no_grad():
            out_ids = eval_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        gen_text = tokenizer.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = None
        try:
            pred = json.loads(strip_fences(gen_text))
            if isinstance(pred, dict):
                metrics['json_valid'] += 1
        except Exception:
            pred = None

        row_ok = {}
        if pred:
            row_ok['title'] = int(norm_str(pred.get('title')) == norm_str(gold.get('title')))
            row_ok['category'] = int(norm_str(pred.get('category')) == norm_str(gold.get('category')))
            row_ok['venue_name'] = int(norm_str(pred.get('venue_name')) == norm_str(gold.get('venue_name')))
            g_dates = sorted({s.get('start_date') for s in gold.get('sessions', []) if s.get('start_date')})
            p_dates = sorted({s.get('start_date') for s in (pred.get('sessions') or []) if isinstance(s, dict) and s.get('start_date')})
            row_ok['dates'] = int(p_dates == g_dates)
            g_sk = [(s.get('start_date'), s.get('start_time'), s.get('end_time')) for s in gold.get('sessions', [])]
            p_sk = [(s.get('start_date'), s.get('start_time'), s.get('end_time')) for s in (pred.get('sessions') or []) if isinstance(s, dict)]
            sf1 = f1_multiset(p_sk, g_sk)
            session_f1s.append(sf1)
            row_ok['sessions_exact'] = int(sorted(str(x) for x in p_sk) == sorted(str(x) for x in g_sk))
            g_tk = [(norm_str(t.get('name')), t.get('price'), t.get('quantity_total')) for t in gold.get('ticket_types', [])]
            p_tk = [(norm_str(t.get('name')), t.get('price'), t.get('quantity_total')) for t in (pred.get('ticket_types') or []) if isinstance(t, dict)]
            tf1 = f1_multiset(p_tk, g_tk)
            ticket_f1s.append(tf1)
            row_ok['tickets_exact'] = int(sorted(str(x) for x in p_tk) == sorted(str(x) for x in g_tk))
            g_rp = gold.get('refund_policy') or {}
            p_rp = pred.get('refund_policy') if isinstance(pred.get('refund_policy'), dict) else {}
            row_ok['refund'] = int(bool(p_rp.get('allow_refunds', False)) == bool(g_rp.get('allow_refunds', False)) and p_rp.get('deadline_days') == g_rp.get('deadline_days'))
            for k, v in row_ok.items():
                metrics[k] += v
        else:
            session_f1s.append(0.0)
            ticket_f1s.append(0.0)

        if gold.get('venue_name') is None:
            null_venue_total += 1
            if pred and pred.get('venue_name') not in (None, ''):
                null_venue_halluc += 1
        if len(gold.get('ticket_types', [])) == 0:
            zero_tkt_total += 1
            if pred and len(pred.get('ticket_types') or []) > 0:
                zero_tkt_halluc += 1

        details.append({'source_file': row.get('source_file'), 'source_group': row.get('source_group'), 'json_ok': bool(pred), 'scores': row_ok})

    n = len(rows)
    summary = {
        'tag': tag,
        'n': n,
        'json_valid_pct': round(100 * metrics['json_valid'] / n, 1),
        'title_acc_pct': round(100 * metrics['title'] / n, 1),
        'category_acc_pct': round(100 * metrics['category'] / n, 1),
        'venue_acc_pct': round(100 * metrics['venue_name'] / n, 1),
        'dates_acc_pct': round(100 * metrics['dates'] / n, 1),
        'sessions_exact_pct': round(100 * metrics['sessions_exact'] / n, 1),
        'sessions_f1_pct': round(100 * sum(session_f1s) / n, 1),
        'tickets_exact_pct': round(100 * metrics['tickets_exact'] / n, 1),
        'tickets_f1_pct': round(100 * sum(ticket_f1s) / n, 1),
        'refund_acc_pct': round(100 * metrics['refund'] / n, 1),
        'null_venue_halluc': f"{null_venue_halluc}/{null_venue_total}",
        'zero_ticket_halluc': f"{zero_tkt_halluc}/{zero_tkt_total}",
        'details': details,
    }
    print(f"=== {tag} (N={n}) ===")
    print(json.dumps({k: v for k, v in summary.items() if k != 'details'}, ensure_ascii=False, indent=2))
    eval_model.config.use_cache = False
    return summary

pre_v2_report = evaluate_model_on_holdout(model, test_rows, tag='PRE_CONTINUED_V2_HOLDOUT')
with open(os.path.join(V3_EXPORT_DIR, 'pre_v2_eval_report.json'), 'w', encoding='utf-8') as f:
    json.dump(pre_v2_report, f, ensure_ascii=False, indent=2)


In [ ]:
# [7] Chạy Continued QLoRA Fine-Tuning (V2 -> V3) & Lưu Adapter + Metadata
from trl import SFTConfig, SFTTrainer

if 'render_prompt' not in globals():
    def render_prompt(messages, add_generation_prompt=False):
        sig = inspect.signature(tokenizer.apply_chat_template)
        kw = {'tokenize': False, 'add_generation_prompt': add_generation_prompt}
        if 'enable_thinking' in sig.parameters:
            kw['enable_thinking'] = False
        return tokenizer.apply_chat_template(messages, **kw)

def filter_valid_kwargs(cls, kwargs):
    sig = inspect.signature(cls.__init__)
    params = sig.parameters
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
        return kwargs
    return {k: v for k, v in kwargs.items() if k in params}

train_ds = Dataset.from_list([{'text': render_prompt(r['messages'], add_generation_prompt=False)} for r in train_rows])
val_ds = Dataset.from_list([{'text': render_prompt(r['messages'], add_generation_prompt=False)} for r in val_rows])

sft_candidates = {
    'output_dir': V3_CHECKPOINT_DIR,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'learning_rate': LEARNING_RATE,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': WARMUP_RATIO,
    'weight_decay': 0.01,
    'max_grad_norm': 0.5,
    'num_train_epochs': NUM_EPOCHS,
    'eval_strategy': 'epoch',
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'fp16': False,  # Tắt GradScaler trên T4 để tránh lỗi BFloat16 unscale; 4-bit NF4 vẫn compute bằng float16 qua BitsAndBytes
    'bf16': BF16_MODE,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'logging_steps': 5,
    'report_to': 'none',
    'seed': SEED,
    'max_length': MAX_SEQ_LENGTH,
    'max_seq_length': MAX_SEQ_LENGTH,
    'dataset_text_field': 'text',
    'packing': False,
}
sft_args = SFTConfig(**filter_valid_kwargs(SFTConfig, sft_candidates))
trainer_candidates = {
    'model': model,
    'train_dataset': train_ds,
    'eval_dataset': val_ds,
    'processing_class': tokenizer,
    'tokenizer': tokenizer,
    'args': sft_args,
}
trainer = SFTTrainer(**filter_valid_kwargs(SFTTrainer, trainer_candidates))
train_res = trainer.train()
eval_res = trainer.evaluate()

trainer.model.save_pretrained(V3_ADAPTER_DIR)
tokenizer.save_pretrained(V3_ADAPTER_DIR)
with open(os.path.join(V3_ADAPTER_DIR, 'continued_training_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'parent_adapter': PARENT_ADAPTER_DIR,
        'base_model': BASE_MODEL_ID,
        'train_samples': len(train_rows),
        'external_replay_added': external_replay_added,
        'train_loss': train_res.training_loss,
        'eval_metrics': eval_res,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ Đã lưu Continued V3 Adapter vào: {V3_ADAPTER_DIR}")


In [ ]:
# [8] Đánh giá Sau Continued Fine-Tuning (V3 vs V2) trên Holdout Test + Kiểm tra Regression cũ
post_v3_report = evaluate_model_on_holdout(trainer.model, test_rows, tag='POST_CONTINUED_V3_HOLDOUT')
with open(os.path.join(V3_EXPORT_DIR, 'post_v3_eval_report.json'), 'w', encoding='utf-8') as f:
    json.dump(post_v3_report, f, ensure_ascii=False, indent=2)

old_replay_rows = [r for r in (train_rows + val_rows + test_rows) if r.get('source_group') == 'old_test_10_replay']
old_regression_report = evaluate_model_on_holdout(trainer.model, old_replay_rows, tag='POST_CONTINUED_V3_OLD10_REGRESSION')
with open(os.path.join(V3_EXPORT_DIR, 'v3_old10_regression_report.json'), 'w', encoding='utf-8') as f:
    json.dump(old_regression_report, f, ensure_ascii=False, indent=2)


In [ ]:
# [9] Merge LoRA V3 -> FP16 -> Convert sang GGUF Q4_K_M & FP16 + Modelfile cho Ollama (Standalone - chạy được ngay sau restart)
!pip uninstall -y -q torchao
import os, json, gc, sys, torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import peft.import_utils

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
V3_ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_v3')
V3_MERGED_DIR = os.path.join(DRIVE_ROOT, 'merged/eventhub_qwen_v3_fp16')
V3_EXPORT_DIR = os.path.join(DRIVE_ROOT, 'export')
os.makedirs(V3_EXPORT_DIR, exist_ok=True)

cfg_path = os.path.join(V3_ADAPTER_DIR, 'adapter_config.json')
if not os.path.isfile(cfg_path):
    raise FileNotFoundError(f"Không tìm thấy Adapter V3 tại {V3_ADAPTER_DIR}")
with open(cfg_path, 'r', encoding='utf-8') as f:
    BASE_MODEL_ID = json.load(f).get('base_model_name_or_path') or 'Qwen/Qwen3-4B'
print(f"✅ Nạp Adapter V3 từ: {V3_ADAPTER_DIR} | Base: {BASE_MODEL_ID}")

for _v in ['trainer', 'model', 'base_model', 'base_fp16', 'merged_model']:
    globals().pop(_v, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
for _m in list(sys.modules.values()):
    if getattr(_m, '__name__', '').startswith('peft') and hasattr(_m, 'is_torchao_available'):
        _m.is_torchao_available = lambda: False

tokenizer = AutoTokenizer.from_pretrained(V3_ADAPTER_DIR, trust_remote_code=True)
# Merge trên CPU (device_map='cpu') để tránh GPU T4 bị offload sang meta device làm hỏng merge
base_fp16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True, trust_remote_code=True)
LOCAL_MERGED_DIR = '/content/v3_merged_fp16'
LOCAL_EXPORT_DIR = '/content/v3_export'
os.makedirs(LOCAL_MERGED_DIR, exist_ok=True)
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

merged_model = PeftModel.from_pretrained(base_fp16, V3_ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(LOCAL_MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(LOCAL_MERGED_DIR)
print(f"✅ Merged FP16 V3 saved locally at: {LOCAL_MERGED_DIR}")

!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp || true
!pip install -q gguf
!cmake -B /content/llama.cpp/build -S /content/llama.cpp && cmake --build /content/llama.cpp/build --target llama-quantize -j 4

# Sửa lỗi transformers mới gặp extra_special_tokens dạng list trong tokenizer_config.json
tok_cfg_path = os.path.join(LOCAL_MERGED_DIR, 'tokenizer_config.json')
if os.path.isfile(tok_cfg_path):
    with open(tok_cfg_path, 'r', encoding='utf-8') as f:
        tok_cfg = json.load(f)
    if isinstance(tok_cfg.get('extra_special_tokens'), list):
        tok_cfg.pop('extra_special_tokens', None)
        with open(tok_cfg_path, 'w', encoding='utf-8') as f:
            json.dump(tok_cfg, f, ensure_ascii=False, indent=2)

local_fp16_gguf = os.path.join(LOCAL_EXPORT_DIR, 'eventhub-extractor-v3-f16.gguf')
local_q8_gguf = os.path.join(LOCAL_EXPORT_DIR, 'eventhub-extractor-v3-Q8_0.gguf')
local_q4_gguf = os.path.join(LOCAL_EXPORT_DIR, 'eventhub-extractor-v3-Q4_K_M.gguf')
!python /content/llama.cpp/convert_hf_to_gguf.py "{LOCAL_MERGED_DIR}" --outfile "{local_fp16_gguf}" --outtype f16
assert os.path.isfile(local_fp16_gguf), f'Chuyển đổi GGUF thất bại, không thấy {local_fp16_gguf}'
!/content/llama.cpp/build/bin/llama-quantize "{local_fp16_gguf}" "{local_q8_gguf}" Q8_0
!/content/llama.cpp/build/bin/llama-quantize "{local_fp16_gguf}" "{local_q4_gguf}" Q4_K_M
assert os.path.isfile(local_q8_gguf) and os.path.isfile(local_q4_gguf), 'Quantize GGUF thất bại!'
!rm -f "{local_fp16_gguf}"  # Xóa file tạm 8GB để Drive nhẹ và không tràn VRAM 8GB

def build_modelfile(gguf_filename):
    return f'''FROM ./{gguf_filename}

TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""

SYSTEM """Bạn là chuyên gia trích xuất thông tin sự kiện của hệ thống EventHub.
Nhiệm vụ của bạn là đọc kỹ văn bản tài liệu sự kiện được cung cấp và trích xuất thông tin theo đúng định dạng JSON tuân thủ schema quy định.

Quy tắc bắt buộc:
1. Chỉ trích xuất thông tin có căn cứ trực tiếp trong tài liệu. Tuyệt đối không bịa đặt hoặc suy diễn thông tin (không hallucination).
2. Các trường không có thông tin hoặc chưa xác định phải gán giá trị null (hoặc mảng rỗng [] đối với tags).
3. Định dạng ngày tháng: YYYY-MM-DD. Định dạng giờ: HH:mm (24 giờ).
4. Phân biệt rõ giờ check-in (checkin_start_time) và giờ bắt đầu sự kiện (start_time).
5. Nếu sự kiện diễn ra nhiều ngày hoặc nhiều phiên, trích xuất đầy đủ từng session trong mảng "sessions".
6. Toàn bộ các hạng vé, giá vé (VNĐ, số nguyên) và số lượng phải được đưa vào mảng "ticket_types". Vé miễn phí ghi price là 0.
7. Chỉ trả về duy nhất chuỗi JSON hợp lệ, bắt đầu bằng {{ và kết thúc bằng }}. Tuyệt đối không thêm lời giải thích hoặc markdown code block bên ngoài."""

PARAMETER num_ctx 6144
PARAMETER num_gpu 99
PARAMETER temperature 0.05
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
'''
with open(os.path.join(LOCAL_EXPORT_DIR, 'Modelfile.extractor_v3'), 'w', encoding='utf-8') as f:
    f.write(build_modelfile('eventhub-extractor-v3-Q4_K_M.gguf'))
with open(os.path.join(LOCAL_EXPORT_DIR, 'Modelfile.extractor_v3_q8'), 'w', encoding='utf-8') as f:
    f.write(build_modelfile('eventhub-extractor-v3-Q8_0.gguf'))

import shutil
os.makedirs(V3_EXPORT_DIR, exist_ok=True)
print('⏳ Đang đồng bộ 2 bản GGUF sang Google Drive...')
shutil.copytree(LOCAL_EXPORT_DIR, V3_EXPORT_DIR, dirs_exist_ok=True)
q8_gguf = os.path.join(V3_EXPORT_DIR, 'eventhub-extractor-v3-Q8_0.gguf')
q4_gguf = os.path.join(V3_EXPORT_DIR, 'eventhub-extractor-v3-Q4_K_M.gguf')
print("🎉 Hoàn tất xuất đủ cả 2 phiên bản GGUF V3:")
print(f"  1. Bản nén Q4_K_M (~2.4 GB, GPU 4GB+)     : {q4_gguf}")
print(f"  2. Bản gần nguyên bản Q8_0 (~4.3 GB, GPU 8GB): {q8_gguf}")
print("Lưu ý: Nếu muốn lưu thêm bản HF FP16 gốc (~8GB) lên Drive, chạy: shutil.copytree(LOCAL_MERGED_DIR, V3_MERGED_DIR, dirs_exist_ok=True)")
